# Regional Operating Points Efficiency Table Consolidation

This notebook scans holdout test evaluation results across all available kinematic regions ($E_T \in [0, 4], \eta \in [0, 4]$) for a target model configuration, selects the best model per region, and evaluates three operating point methods:
1. **Calibrated Hard Cut (Target PD: 94.24%)**
2. **Pileup-Corrected Cut $T(\mu)$ (Target PD)**
3. **Athena Loose Cut**

The results are formatted into a 2D multi-index grid table grouped by $\eta$ and $E_T$ intervals ($P_D [\%]$, $SP [\%]$, $P_F [\%]$), exported to **Parquet (`.parquet`)**, and compiled into a publication-ready **LaTeX (`.tex`)** table matching the reference paper layout.

In [ ]:
from pathlib import Path
import sys

# Resolve repository root
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.core.Validation.RegionalOperatingPointsConsolidator import (
    RegionalOperatingPointsConsolidator,
)

print(f"Project root resolved: {PROJECT_ROOT}")

In [ ]:
# Model & Results Paths
CONFIG_PATH = PROJECT_ROOT / "config/NeuralRinger/ModelV6_HighBatch_newExtraction_20_regions_25Rings.yaml"
RESULTS_PATH = PROJECT_ROOT / "results/config_nameModelV6_HighBatch_newExtraction_20_regions_25Rings_id20260926183510"
MANIFEST_PATH = RESULTS_PATH.parent / "split_manifest.json"

# Evaluation & Benchmark Configuration
TARGET_PD = 0.9424
WORKING_POINT = "loose"       # Athena fast photon cut WP ('loose', 'medium', 'tight')
EFFICIENCIES_CSV = None       # Optional Path to reference efficiencies CSV
BINS_MU = 10
N_BOOTSTRAPS_PILEUP = 100

# Formatting & Display Options
AS_PERCENTAGE = True          # Scale metrics by 100
PRECISION = 2                 # Decimal precision
DECIMAL_COMMA = False         # Use comma as decimal separator (e.g. 94,24)
MATH_DELIMITERS = True        # Wrap cell text with $...$
BOLD_BEST_SP = True           # Bold the highest SP method within each region

# Export Destinations
OUTPUT_DIR = PROJECT_ROOT / "notebooks/Tables"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PARQUET_OUTPUT = OUTPUT_DIR / "operating_points_efficiency.parquet"
LATEX_OUTPUT = OUTPUT_DIR / "operating_points_efficiency.tex"

print(f"Config Path:    {CONFIG_PATH}")
print(f"Results Path:   {RESULTS_PATH}")
print(f"Target PD:      {TARGET_PD*100:.2f}%")
print(f"Athena WP:      {WORKING_POINT}")
print(f"Parquet Output: {PARQUET_OUTPUT}")
print(f"LaTeX Output:   {LATEX_OUTPUT}")

In [ ]:
consolidator = RegionalOperatingPointsConsolidator(
    config_path=CONFIG_PATH,
    results_path=RESULTS_PATH,
    manifest_path=MANIFEST_PATH if MANIFEST_PATH.exists() else None,
    target_pd=TARGET_PD,
    working_point=WORKING_POINT,
    bins_mu=BINS_MU,
    n_bootstraps_pileup=N_BOOTSTRAPS_PILEUP,
    efficiencies_csv=EFFICIENCIES_CSV,
)

# Run evaluation across all valid kinematic regions
records = consolidator.scan_and_evaluate()
print(f"Total regional method records extracted: {len(records)}")

In [ ]:
tidy_df = consolidator.build_tidy_dataframe()
display(tidy_df.head(10))

In [ ]:
grid_table = consolidator.build_grid_table(
    as_percentage=AS_PERCENTAGE,
    precision=PRECISION,
    decimal_comma=DECIMAL_COMMA,
    math_delimiters=MATH_DELIMITERS,
    bold_best_sp=BOLD_BEST_SP,
)

display(grid_table)

In [ ]:
# Save tidy long-form evaluation data to Parquet
saved_parquet_path = consolidator.export_parquet(PARQUET_OUTPUT)
print(f"Tidy performance table saved to Parquet at: {saved_parquet_path}")

# Also export the MultiIndex grid table to Parquet (flattened columns)
grid_parquet_path = OUTPUT_DIR / "operating_points_efficiency_grid.parquet"
consolidator.export_parquet(grid_parquet_path, df=grid_table)
print(f"Grid table saved to Parquet at: {grid_parquet_path}")

In [ ]:
caption = f"Comparative detection efficiency ($P_D$), SP index, and fake rate ($P_F$) across $E_T$ and $|\eta|$ regions for Calibrated Hard Cut, Pileup-Corrected Cut $T(\mu)$, and Athena {WORKING_POINT.capitalize()} Cut."
label = "tab:operating_points_efficiency_comparison"

latex_code = consolidator.to_latex_snippet(
    grid_table=grid_table,
    caption=caption,
    label=label,
    resize_box=True,
    as_percentage=AS_PERCENTAGE,
    precision=PRECISION,
    decimal_comma=DECIMAL_COMMA,
)

saved_latex_path = consolidator.export_latex(LATEX_OUTPUT, latex_code=latex_code)
print(f"LaTeX table exported successfully to: {saved_latex_path}")
print("\n--- LaTeX Code Snippet ---\n")
print(latex_code)